This notebook selects iterativeWGCNA modules associated with BGCs. 

In [6]:
import pandas as pd
from pathlib import Path

base_dir = Path("/Users/annasve/Desktop/article_data/expression_module")

summary_rows = []
total_old_bgcs = set()
total_new_bgcs = set()

all_dfs = []

# keywords of columns to exclude (matches e.g. 216-DNPM-1, 216-ISP2-2, etc.)
EXCLUDE_KEYWORDS = ["DNPM", "ISP2", "MA", "SoyM", "TSB", "gluc", "gly", "malt"]

for strain_dir in base_dir.iterdir():
    if not strain_dir.is_dir():
        continue

    strain = strain_dir.name
    old_bgcs = set()
    new_bgcs = set()

    for file in strain_dir.glob("*borders.xlsx"):
        df = pd.read_excel(file)

        # ---- BGC counting ----
        if "Region_Nr" in df.columns:
            old_bgcs.update(df["Region_Nr"].dropna().astype(str))

        if "refined_BGC_ID" in df.columns:
            new_bgcs.update(df["refined_BGC_ID"].dropna().astype(str))

        # ---- add Strain + Strain_Module ----
        df["Strain"] = strain
        if "Module" in df.columns:
            df["Strain_Module"] = strain + "_" + df["Module"].astype(str)

        # ---- drop expression columns ----
        drop_cols = [
            c for c in df.columns
            if any(k.lower() in c.lower() for k in EXCLUDE_KEYWORDS)
        ]
        df_small = df.drop(columns=drop_cols)

        all_dfs.append(df_small)

    summary_rows.append({
        "Strain": strain,
        "Original_BGCs": len(old_bgcs),
        "Refined_BGCs": len(new_bgcs)
    })

    total_old_bgcs.update(old_bgcs)
    total_new_bgcs.update(new_bgcs)

summary_df = pd.DataFrame(summary_rows).sort_values("Strain")

# Concatenated dataframe (all strains/files)
all_df = pd.concat(all_dfs, ignore_index=True) if all_dfs else pd.DataFrame()

In [7]:
file_path = '/Users/annasve/Desktop/article_data/output/iterative_WGCNA/analysis/fishers_001_refined_purity.xlsx'
bgc_df = pd.read_excel(file_path)

In [8]:
# extract Strain = first two underscore-delimited parts of BGC_ID
bgc_df["Strain"] = (
    bgc_df["BGC_ID"]
    .astype(str)
    .str.split("_")
    .str[:2]
    .str.join("_")
)

# build Strain_Module
bgc_df["Strain_Module"] = (
    bgc_df["Strain"].astype(str) + "_" +
    bgc_df["Module"].astype(str)
)

In [9]:
# collect significant Strain_Module values
sig_modules = set(bgc_df["Strain_Module"].dropna().unique())

# filter all_df to keep only those modules
all_df_filtered = all_df[
    all_df["Strain_Module"].isin(sig_modules)
].copy()


In [10]:
all_df_filtered.columns

Index(['Geneid', 'Product', 'Sequence_Length', 'Gene_Kind', 'Is_Core_Gene',
       'Region_Nr', 'corr_to_bgc', 'expressed_any', 'high_corr',
       'refined_member', 'refined_BGC_ID', 'Module', 'kME', 'Cand_Cluster_1',
       'Cluster_Product_1', 'Cand_Cluster_2', 'Cluster_Product_2',
       'Cand_Cluster_3', 'Cluster_Product_3', 'Cand_Cluster_4',
       'Cluster_Product_4', 'prediction', 'score', 'Description',
       'Preferred_name', 'COG_category', 'PFAMs', 'KEGG_Module', 'Orthogroup',
       'CAI', 'CBI', 'Strain', 'Group_ID', 'Locus_ID', 'Overlap_Lengths',
       'Strain_Module', 'motif_id', 'motif_name', 'start', 'site_sequence',
       'E_value', 'significant_site', 'BGC_region', 'BGC_block', '__gorder__',
       'low_expression_rule', 'Region_Nr_original', 'Cand_Cluster_5',
       'Cluster_Product_5', 'Cand_Cluster_6', 'Cluster_Product_6',
       'Cand_Cluster_7', 'Cluster_Product_7', 'Cand_Cluster_8',
       'Cluster_Product_8', 'Cand_Cluster_9', 'Cluster_Product_9',
       '

In [11]:
all_df_filtered.to_excel('/Users/annasve/Desktop/article_data/output/iterative_WGCNA/analysis/bgc_modules_refined.xlsx', index = False)

In [13]:
import numpy as np
import pandas as pd

df = all_df.copy()  # <- replace

# --- unassigned definition: EXACTLY "UNCLASSIFIED" ---
def is_unassigned_module(x):
    if pd.isna(x):
        return True
    return str(x).strip().upper() == "UNCLASSIFIED"

df["Strain"] = df["Strain"].astype(str)
df["Module"] = df["Module"].astype(str)

df["is_unassigned"] = df["Module"].apply(is_unassigned_module)

# -------------------------
# 1) Modules per strain (excluding UNCLASSIFIED)
# -------------------------
modules_per_strain = (
    df.loc[~df["is_unassigned"], ["Strain", "Module"]]
      .drop_duplicates()
      .groupby("Strain")["Module"]
      .nunique()
      .rename("n_modules")
      .reset_index()
)

# -------------------------
# 2) Unclassified genes per strain
# -------------------------
# If Geneid is unique per gene, use nunique to avoid double-counting
unclassified_genes_per_strain = (
    df.loc[df["is_unassigned"]]
      .groupby("Strain")["Geneid"]
      .nunique()
      .rename("n_unclassified_genes")
      .reset_index()
)

# also useful: fraction unclassified per strain
total_genes_per_strain = (
    df.groupby("Strain")["Geneid"]
      .nunique()
      .rename("n_total_genes")
      .reset_index()
)

unclassified_summary = (
    total_genes_per_strain
      .merge(unclassified_genes_per_strain, on="Strain", how="left")
)
unclassified_summary["n_unclassified_genes"] = unclassified_summary["n_unclassified_genes"].fillna(0).astype(int)
unclassified_summary["frac_unclassified"] = (
    unclassified_summary["n_unclassified_genes"] / unclassified_summary["n_total_genes"]
)

# -------------------------
# 3) Combine per-strain summary table
# -------------------------
wgcna_summary_per_strain = (
    modules_per_strain
      .merge(unclassified_summary, on="Strain", how="outer")
      .sort_values("n_modules", ascending=False)
)

# -------------------------
# Print headline stats
# -------------------------
print("Modules per strain summary (excluding UNCLASSIFIED):")
print(wgcna_summary_per_strain["n_modules"].describe())

print("\nUnclassified genes per strain summary:")
print(wgcna_summary_per_strain["n_unclassified_genes"].describe())

print("\nFraction unclassified per strain summary:")
print(wgcna_summary_per_strain["frac_unclassified"].describe())

# wgcna_summary_per_strain is ready for a Supplementary Table
wgcna_summary_per_strain.to_csv("wgcna_summary_per_strain.csv", index=False)


Modules per strain summary (excluding UNCLASSIFIED):
count    132.000000
mean     119.742424
std       44.939930
min       28.000000
25%       90.750000
50%      109.000000
75%      137.000000
max      296.000000
Name: n_modules, dtype: float64

Unclassified genes per strain summary:
count     132.000000
mean      544.704545
std       536.554057
min       101.000000
25%       228.000000
50%       273.000000
75%       826.500000
max      3640.000000
Name: n_unclassified_genes, dtype: float64

Fraction unclassified per strain summary:
count    132.000000
mean       0.066748
std        0.061449
min        0.018919
25%        0.029272
50%        0.034811
75%        0.095814
max        0.376773
Name: frac_unclassified, dtype: float64


In [14]:

# wgcna_summary_per_strain is ready for a Supplementary Table
wgcna_summary_per_strain.to_excel("wgcna_summary_per_strain.xlsx", index=False)

In [16]:
all_df_filtered.columns

Index(['Geneid', 'Product', 'Sequence_Length', 'Gene_Kind', 'Is_Core_Gene',
       'Region_Nr', 'corr_to_bgc', 'expressed_any', 'high_corr',
       'refined_member', 'refined_BGC_ID', 'Module', 'kME', 'Cand_Cluster_1',
       'Cluster_Product_1', 'Cand_Cluster_2', 'Cluster_Product_2',
       'Cand_Cluster_3', 'Cluster_Product_3', 'Cand_Cluster_4',
       'Cluster_Product_4', 'prediction', 'score', 'Description',
       'Preferred_name', 'COG_category', 'PFAMs', 'KEGG_Module', 'Orthogroup',
       'CAI', 'CBI', 'Strain', 'Group_ID', 'Locus_ID', 'Overlap_Lengths',
       'Strain_Module', 'motif_id', 'motif_name', 'start', 'site_sequence',
       'E_value', 'significant_site', 'BGC_region', 'BGC_block', '__gorder__',
       'low_expression_rule', 'Region_Nr_original', 'Cand_Cluster_5',
       'Cluster_Product_5', 'Cand_Cluster_6', 'Cluster_Product_6',
       'Cand_Cluster_7', 'Cluster_Product_7', 'Cand_Cluster_8',
       'Cluster_Product_8', 'Cand_Cluster_9', 'Cluster_Product_9',
       '

In [17]:
n_strain_modules = all_df_filtered["Strain_Module"].nunique(dropna=True)
print("Total unique Strain_Module in all_df_filtered:", n_strain_modules)


Total unique Strain_Module in all_df_filtered: 895


In [18]:
n_strain_modules = all_df["Strain_Module"].nunique(dropna=True)
print("Total unique Strain_Module in all_df:", n_strain_modules)


Total unique Strain_Module in all_df: 15938
